# Projet End-to-End Computer Vision

Classification d'images avec CNN (Convolutional Neural Networks) - Pipeline complet de Machine Learning

## 1. Installation des dépendances

In [ ]:
import subprocess
import sys

# Installation des packages nécessaires
packages = ['torch', 'torchvision', 'matplotlib', 'numpy', 'scikit-learn', 'pillow']

for package in packages:
    try:
        __import__(package)
    except ImportError:
        print(f"Installation de {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

print("✓ Toutes les dépendances sont installées!")

## 2. Imports et Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device utilisé: {DEVICE}")

BATCH_SIZE = 128
LEARNING_RATE = 0.001
EPOCHS = 10
NUM_CLASSES = 10

## 3. Chargement et Visualisation du Dataset (MNIST)

In [ ]:
# Définir les transformations
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # Moyennes et écarts-types MNIST
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Charger MNIST
print("Téléchargement du dataset MNIST...")
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=train_transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=test_transform)

print(f"✓ Dataset MNIST chargé")
print(f"  - Entraînement: {len(train_dataset)} images")
print(f"  - Test: {len(test_dataset)} images")
print(f"  - Dimension des images: {train_dataset[0][0].shape}")

## 4. Visualisation d'exemples

In [ ]:
# Afficher quelques exemples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()

for i in range(10):
    image, label = train_dataset[i]
    axes[i].imshow(image.squeeze(), cmap='gray')
    axes[i].set_title(f'Label: {label}')
    axes[i].axis('off')

plt.suptitle('Exemples d\'images MNIST')
plt.tight_layout()
plt.show()

## 5. Créer les DataLoaders

In [ ]:
# Créer les DataLoaders avec data augmentation
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(f"✓ DataLoaders créés")
print(f"  - Batches d'entraînement: {len(train_loader)}")
print(f"  - Batches de test: {len(test_loader)}")

## 6. Architecture du Modèle CNN

In [ ]:
class CNNModel(nn.Module):
    def __init__(self, num_classes=10):
        super(CNNModel, self).__init__()
        
        # Block 1
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        # Block 2
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(128)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        # Fully Connected Layers
        self.fc1 = nn.Linear(128 * 7 * 7, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, num_classes)
        
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        # Block 1
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        
        # Block 2
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # FC Layers
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

# Créer le modèle
model = CNNModel(num_classes=NUM_CLASSES).to(DEVICE)
print(f"✓ Modèle CNN créé")
print(f"\nArchitecture du modèle:")
print(model)

## 7. Nombre de paramètres

In [ ]:
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params

num_params = count_parameters(model)
print(f"Nombre total de paramètres trainables: {num_params:,}")

## 8. Définir la Loss et l'Optimizer

In [ ]:
# Loss function et optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print(f"✓ Loss function: CrossEntropyLoss")
print(f"✓ Optimizer: Adam (lr={LEARNING_RATE})")
print(f"✓ Scheduler: StepLR (step_size=5, gamma=0.5)")

## 9. Fonctions d'Entraînement et Évaluation

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Statistics
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total
    
    return avg_loss, accuracy

def evaluate(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(test_loader)
    accuracy = 100 * correct / total
    
    return avg_loss, accuracy, np.array(all_preds), np.array(all_labels)

print("✓ Fonctions d'entraînement et d'évaluation définies")

## 10. Boucle d'Entraînement

In [ ]:
# Listes pour tracker la performance
train_losses = []
train_accs = []
val_losses = []
val_accs = []

print(f"Début de l'entraînement sur {EPOCHS} epochs...\n")

for epoch in range(EPOCHS):
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # Validate
    val_loss, val_acc, _, _ = evaluate(model, test_loader, criterion, DEVICE)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    # Update learning rate
    scheduler.step()
    
    print(f"Epoch [{epoch+1}/{EPOCHS}]")
    print(f"  Train - Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%")
    print(f"  Val   - Loss: {val_loss:.4f}, Accuracy: {val_acc:.2f}%\n")

print("✓ Entraînement terminé!")

## 11. Visualisation des Courbes de Performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss
axes[0].plot(train_losses, label='Train Loss', linewidth=2)
axes[0].plot(val_losses, label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Courbe de Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(train_accs, label='Train Accuracy', linewidth=2)
axes[1].plot(val_accs, label='Val Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Courbe de Précision')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nRésumé de la performance:")
print(f"  Train Accuracy finale: {train_accs[-1]:.2f}%")
print(f"  Val Accuracy finale: {val_accs[-1]:.2f}%")

## 12. Évaluation Détaillée et Matrice de Confusion

In [ ]:
# Obtenir les prédictions finales
_, final_acc, all_preds, all_labels = evaluate(model, test_loader, criterion, DEVICE)

# Matrice de confusion
cm = confusion_matrix(all_labels, all_preds)

# Visualiser la matrice de confusion
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True)
plt.xlabel('Prédiction')
plt.ylabel('Vrai Label')
plt.title('Matrice de Confusion')
plt.tight_layout()
plt.show()

print("\nRapport de Classification:")
print(classification_report(all_labels, all_preds, target_names=[str(i) for i in range(10)]))

## 13. Visualisation des Prédictions

In [ ]:
# Afficher 20 prédictions
model.eval()
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
axes = axes.flatten()

with torch.no_grad():
    images, labels = next(iter(test_loader))
    images, labels = images.to(DEVICE), labels.to(DEVICE)
    
    outputs = model(images)
    _, predictions = torch.max(outputs, 1)
    
    for i in range(20):
        image = images[i].cpu().squeeze()
        pred = predictions[i].item()
        true = labels[i].item()
        
        axes[i].imshow(image, cmap='gray')
        color = 'green' if pred == true else 'red'
        axes[i].set_title(f'Vrai: {true}, Pred: {pred}', color=color)
        axes[i].axis('off')

plt.suptitle('Prédictions du Modèle')
plt.tight_layout()
plt.show()

## 14. Sauvegarde du Modèle

In [ ]:
# Sauvegarder le modèle
model_path = 'mnist_cnn_model.pth'
torch.save(model.state_dict(), model_path)

print(f"✓ Modèle sauvegardé: {model_path}")

# Vérifier que le modèle peut être rechargé
loaded_model = CNNModel(num_classes=NUM_CLASSES).to(DEVICE)
loaded_model.load_state_dict(torch.load(model_path, map_location=DEVICE))
print(f"✓ Modèle rechargé avec succès")

## 15. Prédiction sur une Image Personnalisée

In [ ]:
def predict_single_image(model, image, device):
    """Faire une prédiction sur une seule image"""
    model.eval()
    
    with torch.no_grad():
        # Ajouter batch dimension
        image = image.unsqueeze(0).to(device)
        
        # Forward pass
        output = model(image)
        probabilities = torch.softmax(output, dim=1)
        
        pred_class = torch.argmax(probabilities, dim=1).item()
        confidence = probabilities[0, pred_class].item()
    
    return pred_class, confidence, probabilities[0].cpu().numpy()

# Exemple de prédiction
test_image, test_label = test_dataset[42]
pred_class, confidence, probs = predict_single_image(model, test_image, DEVICE)

# Afficher le résultat
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Image
axes[0].imshow(test_image.squeeze(), cmap='gray')
axes[0].set_title(f'Vrai Label: {test_label}\nPrédiction: {pred_class} (Confiance: {confidence:.2%})')
axes[0].axis('off')

# Distribution des probabilités
axes[1].bar(range(10), probs)
axes[1].set_xlabel('Classe')
axes[1].set_ylabel('Probabilité')
axes[1].set_title('Distribution des Probabilités')
axes[1].set_xticks(range(10))

plt.tight_layout()
plt.show()

## 16. Analyse des Erreurs

In [ ]:
# Trouver les images mal classées
model.eval()
misclassified_indices = []

with torch.no_grad():
    for idx, (image, label) in enumerate(test_dataset):
        pred_class, _, _ = predict_single_image(model, image, DEVICE)
        if pred_class != label:
            misclassified_indices.append(idx)
            if len(misclassified_indices) >= 12:
                break

# Afficher les images mal classées
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.flatten()

for i, idx in enumerate(misclassified_indices[:12]):
    image, label = test_dataset[idx]
    pred_class, confidence, _ = predict_single_image(model, image, DEVICE)
    
    axes[i].imshow(image.squeeze(), cmap='gray')
    axes[i].set_title(f'Vrai: {label}, Pred: {pred_class}\nConfiance: {confidence:.1%}', color='red')
    axes[i].axis('off')

plt.suptitle('Exemples d\'Erreurs du Modèle')
plt.tight_layout()
plt.show()

print(f"Nombre d'images mal classées: {len([i for i in range(len(test_dataset)) if predict_single_image(model, test_dataset[i][0], DEVICE)[0] != test_dataset[i][1]])}")

## 17. Résumé et Points Clés

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════╗
║              PIPELINE COMPUTER VISION COMPLET             ║
╚═══════════════════════════════════════════════════════════╝

✓ ÉTAPES COUVERTES:
  1. Chargement et prétraitement des données
  2. Visualisation du dataset
  3. Architecture CNN personnalisée
  4. Entraînement avec validation
  5. Évaluation complémentaire
  6. Visualisation des résultats
  7. Matrice de confusion et métriques
  8. Sauvegarde/rechargement du modèle
  9. Inférence sur nouvelles images
  10. Analyse des erreurs

📊 RÉSULTATS FINAUX:
  - Accuracy Test: {:.2f}%
  - Nombre de paramètres: {:,}
  - Epochs d'entraînement: {}
  - Device: {}

🔧 CONCEPTS CLÉS APRIS:
  • Préparation des données avec transforms
  • Architecture CNN (convolutions, pooling)
  • Batch normalization et dropout
  • Entraînement et validation
  • Optimisation avec schedulers
  • Évaluation et métriques
  • Visualisation et analyse des résultats

💡 AMÉLIORATIONS POSSIBLES:
  • Data augmentation (rotations, zoom, etc.)
  • Hyperparameter tuning
  • Transfer learning (VGG, ResNet, etc.)
  • Ensemble methods
  • Techniques de régularisation avancées
""".format(val_accs[-1], num_params, EPOCHS, DEVICE))